# V102 - Blend of bagged views

Pre-registered plan: blends of single views failed, but bagged views have lower variance, so their combination may behave differently. OOF-weighted average of bagged word, assessment, and mentions faces bagged word-only under grouped folds with a fresh seed. Same locks. If variance was what poisoned blends, this passes.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 2727
N_SPLITS = 5
GATE_FLOOR = 0.002
BAGS = 10
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v102")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Bag each view, weight by OOF, gate the blend

Weights are OOF means, fixed before seeing the blend verdict. Worst fold locks as usual.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"])

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
low = [d.lower() for d in docs]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
STOP = set("dengan yang dan di ke dari untuk pada adalah ini itu sebagai dalam juga akan ada".split())
MN = np.zeros((n_train, len(MODS)))
for j, m in enumerate(MODS):
    row = catalog.set_index("module_id").loc[m] if m in set(catalog["module_id"]) else None
    keys = set(re.findall(r"[a-z]+", (str(row["module_name"]) + " " + str(row["description_and_syllabus"])).lower())) - STOP if row is not None else set()
    keys = {k for k in keys if len(k) > 3}
    for i, d in enumerate(low):
        MN[i, j] = sum(d.count(k) for k in keys)
MN = (MN - MN.mean(0)) / (MN.std(0) + 1e-9)
VIEWS = {"word": W, "assess": A, "mentions": np.nan_to_num(MN)}

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in all_users])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def bagged_oof(Xtr, seed):
    r = np.random.RandomState(seed)
    acc = np.zeros_like(Y)
    cnt = np.zeros(n_train)
    for b in range(BAGS):
        idx = r.choice(n_train, n_train, replace=True)
        oob = np.setdiff1d(np.arange(n_train), np.unique(idx))
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(Xtr[idx], Y[idx, j])
            acc[oob, j] += m.predict(Xtr[oob])
        cnt[oob] += 1
    return acc / np.maximum(cnt[:, None], 1)

oof = {k: bagged_oof(V, SEED + i) for i, (k, V) in enumerate(VIEWS.items())}
for k, o in oof.items():
    print(f"{k}: {ndcg5(Y, o).mean():.5f}")
w = np.array([max(ndcg5(Y, o).mean(), 0) for o in oof.values()])
w = w / w.sum()
blend = sum(wi * o for wi, o in zip(w, oof.values()))
base = ndcg5(Y, oof["word"])
d = ndcg5(Y, blend) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
print(f"blend OOF={ndcg5(Y, blend).mean():.5f} weights={dict(zip(oof, np.round(w, 3)))} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}]")
print("ADOPT bagged blend" if lo > GATE_FLOOR else "KEEP bagged word; bagging does not rescue blends")